In [1]:
import pandas as pd
import os
import shutil

In [2]:
destination_dir = '../data/baseline_load_profiles/raw'
os.makedirs("../data/baseline_load_profiles/raw", exist_ok=True)

In [3]:
caiso_subba_load_profiles = pd.read_csv(
    "../data/iso_load_profiles/caiso.csv",
    parse_dates=['timestamp']
)
caiso_subba_forecast_profiles = pd.read_csv(
    "../data/iso_load_profiles/caiso_forecast.csv",
    parse_dates=['timestamp']
)

ercot_subba_load_profiles = pd.read_csv(
    "../data/iso_load_profiles/ercot.csv",
    parse_dates=['timestamp']
)
ercot_subba_forecast_profiles = pd.read_csv(
    "../data/iso_load_profiles/ercot_forecast.csv",
    parse_dates=['timestamp']
)

isone_subba_load_profiles = pd.read_csv(
    "../data/iso_load_profiles/isone.csv",
    parse_dates=['timestamp']
)
isone_subba_load_profiles['subba'] = isone_subba_load_profiles['subba'].astype(str)
isone_subba_forecast_profiles = pd.read_csv(
    "../data/iso_load_profiles/isone_forecast.csv",
    parse_dates=['timestamp']
)
isone_subba_forecast_profiles['subba'] = isone_subba_forecast_profiles['subba'].astype(str)

miso_subba_load_profiles = pd.read_csv(
    "../data/iso_load_profiles/miso.csv",
    parse_dates=['timestamp']
)
miso_subba_load_profiles['subba'] = miso_subba_load_profiles['subba'].astype(str).str.zfill(4)
miso_subba_forecast_profiles = pd.read_csv(
    "../data/iso_load_profiles/miso_forecast.csv",
    parse_dates=['timestamp']
)
miso_subba_forecast_profiles['subba'] = miso_subba_forecast_profiles['subba'].astype(str).str.zfill(4)

pjm_subba_load_profiles = pd.read_csv(
    "../data/iso_load_profiles/pjm.csv",
    parse_dates=['timestamp']
)
pjm_subba_forecast_profiles = pd.read_csv(
    "../data/iso_load_profiles/pjm_forecast.csv",
    parse_dates=['timestamp']
)

nyiso_subba_load_profiles = pd.read_csv(
    "../data/iso_load_profiles/nyiso.csv",
    parse_dates=['timestamp']
)
nyiso_subba_forecast_profiles = pd.read_csv(
    "../data/iso_load_profiles/nyiso_forecast.csv",
    parse_dates=['timestamp']
)

spp_subba_load_profiles = pd.read_csv(
    "../data/iso_load_profiles/spp.csv",
    parse_dates=['timestamp']
)

In [4]:
subba_load_profiles_pre_2019 = (
    pd.concat(
        [
            caiso_subba_load_profiles,
            ercot_subba_load_profiles,
            isone_subba_load_profiles,
            miso_subba_load_profiles,
            pjm_subba_load_profiles,
            nyiso_subba_load_profiles,
            spp_subba_load_profiles,
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

subba_forecast_profiles = (
    pd.concat(
        [
            caiso_subba_forecast_profiles,
            ercot_subba_forecast_profiles,
            isone_subba_forecast_profiles,
            miso_subba_forecast_profiles,
            nyiso_subba_forecast_profiles,
            pjm_subba_forecast_profiles,
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

In [5]:
import matplotlib.pyplot as plt

In [7]:
df_erco_replacement = (
    ercot_subba_load_profiles.loc[ercot_subba_load_profiles.timestamp.dt.year == 2019]
    .copy()
)

df_reco_replacement = (
    pjm_subba_load_profiles.loc[pjm_subba_load_profiles.subba == 'RECO']
    .copy()
)

df_pl_replacement = pjm_subba_load_profiles.loc[pjm_subba_load_profiles.subba == 'PL']
df_pl_replacement = (
    df_pl_replacement.loc[(
        (df_pl_replacement.timestamp.dt.year == 2019)
        & (df_pl_replacement.timestamp.dt.month == 1)
    )]
)

df_pgae_replacement = caiso_subba_load_profiles.loc[caiso_subba_load_profiles.subba == 'PGAE']
df_pgae_replacement = (
    df_pgae_replacement.loc[(
        (df_pgae_replacement.timestamp >= '2019-01-01 00:00:00')
        & (df_pgae_replacement.timestamp <= '2019-05-02 07:00:00')
    )]
)

df_vea_replacement = caiso_subba_load_profiles.loc[caiso_subba_load_profiles.subba == 'VEA']
df_vea_replacement = (
    df_vea_replacement.loc[(
        (df_vea_replacement.timestamp >= '2019-12-23 09:00:00')
        & (df_vea_replacement.timestamp <= '2020-05-02 07:00:00')
    )]
)

df_replacement = (
    pd.concat([
        df_erco_replacement,
        df_reco_replacement,
        df_pl_replacement,
        df_pgae_replacement,
        df_vea_replacement
    ],
    ignore_index=True)
    .rename(columns={'value': 'new_value'})
)

In [8]:
eia_930_ref_bas = pd.read_excel('../data/EIA930_Reference_Tables.xlsx', sheet_name='BAs')
ba_code_name_map = dict(zip(
    eia_930_ref_bas['BA Code'],
    eia_930_ref_bas['BA Name']
))

eia_930_ref_subbas = pd.read_excel('../data/EIA930_Reference_Tables.xlsx', sheet_name='BA Subregions')
subba_code_name_map = dict(zip(
    eia_930_ref_subbas['BA Subregion Code'],
    eia_930_ref_subbas['BA Subregion Name']
))
subba_parent_map = dict(zip(
    eia_930_ref_subbas['BA Subregion Code'],
    eia_930_ref_subbas['BA Code']
))

In [9]:
for year in range(2016, 2025):
    # Sub-BA load profiles
    if year < 2019:
        df = (
            subba_load_profiles_pre_2019.loc[subba_load_profiles_pre_2019.timestamp.dt.year == year]
            .copy()
        )
    else:
        eia_subba_profile_fpath = f"../data/eia_load_profiles/{year}_hourly_demand_by_subregion.csv"
        df = pd.read_csv(eia_subba_profile_fpath, parse_dates=['timestamp'])
        _df_replacement = df_replacement.loc[df_replacement.timestamp.dt.year == year]
        df = df.merge(_df_replacement, on=['timestamp', 'subba'], how='outer')
        df.loc[df.new_value.notna(), 'value'] = (
            df.loc[df.new_value.notna(), 'new_value'].round().astype(int)
        )
        df = df.drop(columns='new_value')

    df['subba'] = df['subba'].str.strip()
    df['subba-name'] = df['subba'].map(subba_code_name_map)
    df['parent'] = df['subba'].map(subba_parent_map)
    df['parent-name'] = df['parent'].map(ba_code_name_map)
    df = (
        df.loc[df.parent != 'PNM']
        .dropna(subset='subba-name')
        .drop_duplicates(['subba', 'timestamp'])
        .sort_values('timestamp')
    )
    df.to_csv(os.path.join(destination_dir, f"{year}_hourly_demand_by_subregion.csv"), index=False)

    # Sub-BA forecast profiles
    df = (
        subba_forecast_profiles.loc[subba_forecast_profiles.timestamp.dt.year == year]
        .copy()
    )
    df['subba'] = df['subba'].str.strip()
    df['subba-name'] = df['subba'].map(subba_code_name_map)
    df['parent'] = df['subba'].map(subba_parent_map)
    df['parent-name'] = df['parent'].map(ba_code_name_map)
    df = (
        df.drop_duplicates(['subba', 'timestamp'])
        .sort_values('timestamp')
    )
    df.to_csv(os.path.join(destination_dir, f"{year}_hourly_forecast_by_subregion.csv"), index=False)

    # BA load profiles
    eia_ba_profile_fpath = f"../data/eia_load_profiles/{year}_hourly_demand_by_rto.csv"
    df = pd.read_csv(eia_ba_profile_fpath, parse_dates=['timestamp'])
    df['respondent-name'] = df['respondent'].map(ba_code_name_map)
    df = (
        df.dropna(subset='respondent-name')
        .drop_duplicates(['respondent', 'timestamp'])
        .sort_values('timestamp')
    )
    df.to_csv(os.path.join(destination_dir, f"{year}_hourly_demand_by_rto.csv"), index=False)

    # BA forecast profiles
    eia_ba_forecast_profile_fpath = f"../data/eia_load_profiles/{year}_hourly_forecast_by_rto.csv"
    df = pd.read_csv(eia_ba_forecast_profile_fpath, parse_dates=['timestamp'])
    df['respondent-name'] = df['respondent'].map(ba_code_name_map)
    df = (
        df.dropna(subset='respondent-name')
        .drop_duplicates(['respondent', 'timestamp'])
        .sort_values('timestamp')
    )
    df.to_csv(os.path.join(destination_dir, f"{year}_hourly_forecast_by_rto.csv"), index=False)